<a href="https://www.kaggle.com/code/abdullahalmamunzihan/simplegan?scriptVersionId=292154576" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Simple GAN architecture to understand how actually GAN works.

### https://github.com/abdullahzihan16/simpleGAN

## Install PESQ and STOI

In [ ]:
!pip install pesq pystoi


## import and config

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from scipy.io import wavfile
import os
import random
import matplotlib.pyplot as plt
from scipy.signal import stft
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Do your Change Here
epochs = 15 
batch_size = 32
lr=0.0002
l1_lambda=100
num_wav_file = 100 # number of training wav file
train_dir = r'/kaggle/input/dns-challenge-simple-gan/training_samples/train' # replace according to your path
test_file = r'/kaggle/input/dns-challenge-simple-gan/training_samples/dev/audio_10.wav'# replace according to your path


In [ ]:

def load_audio(path):
    x, sr = sf.read(path, always_2d=True)
    return x, sr

def visualize_audio(x, sr, title, max_seconds=5):
    n_samples, n_ch = x.shape
    max_n = min(n_samples, int(max_seconds * sr))
    t = np.arange(max_n) / sr
    plt.figure(figsize=(12, 2.6 * n_ch))
    for ch in range(n_ch):
        ax = plt.subplot(n_ch, 1, ch + 1)
        ax.plot(t, x[:max_n, ch])
        ax.set_ylabel(f"Ch {ch+1}")
        if ch == 0:
            ax.set_title(title)
        if ch == n_ch - 1:
            ax.set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()

def play_audio(x, sr, label):
    print(f"Playing: {label}")
    display(Audio(data=x, rate=sr))

def list_wavs(folder):
    wavs = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".wav"):
                wavs.append(os.path.join(root, f))
    return wavs


train_wavs = list_wavs(train_dir)
if len(train_wavs) == 0:
    raise FileNotFoundError(f"No .wav files found inside train_dir: {train_dir}")

random.shuffle(train_wavs)
picked_train = train_wavs[:2]  # random 2 training audios

picked_files = picked_train + [test_file]

# Show info, visualize and play
for idx, path in enumerate(picked_files, start=1):
    x, sr = load_audio(path)
    name = Path(path).name

    print("-" * 60)
    print(f"Audio {idx}: {name}")
    print(f"Path        : {path}")
    print(f"Sample rate  : {sr} Hz")
    print(f"Channels     : {x.shape[1]}")
    print(f"Duration     : {x.shape[0]/sr:.2f} sec")

    # if dual-channel (noisy/clean)
    if x.shape[1] == 2:
        title = f"{name} (2 channels: Ch1=noisy, Ch2=clean)"
    else:
        title = f"{name} ({x.shape[1]} channel(s))"

    visualize_audio(x, sr, title=title, max_seconds=5)

    # For dual-channel training wav: play noisy and clean separately too
    if x.shape[1] == 2:
        play_audio(x[:, 0], sr, f"{name} - Ch1 (noisy)")
        play_audio(x[:, 1], sr, f"{name} - Ch2 (clean)")
    else:
        play_audio(x, sr, f"{name}")


## Dataloder


### It loads paired noisy–clean speech from dual-channel WAV files, converts them to tensors, and returns fixed-length audio segments for training.

In [ ]:
class SpeechDataset(Dataset):
    def __init__(self, file_list, segment_length=16384):
        self.files = file_list
        self.segment_length = segment_length

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fs, data = wavfile.read(self.files[idx])
        if fs != 16000:
            raise ValueError("Sample rate must be 16 kHz.")
        
        noisy = data[:, 0].astype(np.float32) / 32768.0
        clean = data[:, 1].astype(np.float32) / 32768.0
        
        if len(noisy) > self.segment_length:
            start = np.random.randint(0, len(noisy) - self.segment_length)
            noisy = noisy[start:start + self.segment_length]
            clean = clean[start:start + self.segment_length]
        else:
            pad = self.segment_length - len(noisy)
            noisy = np.pad(noisy, (0, pad))
            clean = np.pad(clean, (0, pad))
        
        return torch.tensor(noisy), torch.tensor(clean)

## Generator

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        kernel_size = 31
        stride = 2
        padding = 15
        output_padding = 1
        
        enc_in_ch = [1, 16, 32, 32, 64, 64, 128, 128, 256, 256, 512]
        enc_out_ch = [16, 32, 32, 64, 64, 128, 128, 256, 256, 512, 1024]
        
        self.enc_convs = nn.ModuleList([
            nn.Conv1d(enc_in_ch[i], enc_out_ch[i], kernel_size, stride, padding)
            for i in range(11)
        ])
        self.enc_prelus = nn.ModuleList([
            nn.PReLU(enc_out_ch[i]) for i in range(11)
        ])
        
        dec_in_ch = [2048, 1024, 512, 512, 256, 256, 128, 128, 64, 64, 32]
        dec_out_ch = [512, 256, 256, 128, 128, 64, 64, 32, 32, 16, 1]
        
        self.dec_convs = nn.ModuleList([
            nn.ConvTranspose1d(dec_in_ch[i], dec_out_ch[i], kernel_size, stride, padding, output_padding)
            for i in range(11)
        ])
        self.dec_acts = nn.ModuleList(
            [nn.PReLU(dec_out_ch[i]) for i in range(10)] + [nn.Tanh()]
        )

    def forward(self, x, z=None):
        e = x.unsqueeze(1)
        es = []
        for i in range(11):
            e = self.enc_convs[i](e)
            e = self.enc_prelus[i](e)
            es.append(e)
        
        if z is None:
            z = torch.randn_like(es[-1])
        
        d = torch.cat([es[-1], z], dim=1)
        
        for j in range(11):
            d = self.dec_convs[j](d)
            d = self.dec_acts[j](d)
            if j < 10:
                d = torch.cat([d, es[9 - j]], dim=1) 
        
        return d.squeeze(1)

## Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_length=16384):
        super(Discriminator, self).__init__()
        kernel_size = 31
        stride = 2
        padding = 15
        
        d_in_ch = [2, 16, 32, 32, 64, 64, 128, 128, 256, 256, 512]
        d_out_ch = [16, 32, 32, 64, 64, 128, 128, 256, 256, 512, 1024]
        
        self.convs = nn.ModuleList([
            nn.Conv1d(d_in_ch[i], d_out_ch[i], kernel_size, stride, padding)
            for i in range(11)
        ])
        self.bns = nn.ModuleList([
            nn.BatchNorm1d(d_out_ch[i]) for i in range(11)
        ])
        self.acts = nn.ModuleList([
            nn.LeakyReLU(0.3) for _ in range(11)
        ])
        
        final_dim = d_out_ch[-1] * (input_length // (2 ** 11))  # 1024 * 8
        self.fc = nn.Linear(final_dim, 1)

    def forward(self, noisy, target):
        x = torch.cat([noisy.unsqueeze(1), target.unsqueeze(1)], dim=1)
        for i in range(11):
            x = self.convs[i](x)
            x = self.bns[i](x)
            x = self.acts[i](x)
        x = x.view(x.size(0), -1)
        return self.fc(x)  # Raw output for LSGAN

## Training

In [ ]:
# Training function (LSGAN losses)
def train_gan(train_dir):
    global epochs, batch_size, lr, l1_lambda, num_wav_file
    file_list = [os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith('.wav')]
    
    random.shuffle(file_list)

    if num_wav_file < len(file_list):
        file_list = file_list[:num_wav_file]
        print(f"Using {num_wav_file} randomly selected files for training.")
    else:
        print(f"Using all {len(file_list)} available files.")

    
    dataset = SpeechDataset(file_list)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    G = Generator().to(device)
    D = Discriminator().to(device)
    
    optimizer_g = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optimizer_d = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.MSELoss()
    
    log_file = open("training_log.txt", "w")
    log_file.write("Epoch,D_Loss,G_Loss\n")
    
    for epoch in range(epochs):
        epoch_d_loss = 0.0
        epoch_g_loss = 0.0
        num_batches = 0
        
        for noisy, clean in dataloader:
            noisy, clean = noisy.to(device), clean.to(device)
            batch_size = noisy.size(0)
            
            z = torch.randn(batch_size, 1024, 8).to(device)
            
            # Train Discriminator
            optimizer_d.zero_grad()
            gen = G(noisy, z)
            real_out = D(noisy, clean)
            fake_out = D(noisy, gen.detach())
            
            real_label = torch.ones_like(real_out).to(device)
            fake_label = torch.zeros_like(fake_out).to(device)
            
            loss_d_real = 0.5 * criterion(real_out, real_label)
            loss_d_fake = 0.5 * criterion(fake_out, fake_label)
            loss_d = loss_d_real + loss_d_fake
            loss_d.backward()
            optimizer_d.step()
            
            # Train Generator
            optimizer_g.zero_grad()
            fake_out = D(noisy, gen)
            loss_g_adv = criterion(fake_out, torch.ones_like(fake_out).to(device))
            loss_g_l1 = F.l1_loss(gen, clean) * l1_lambda
            loss_g = loss_g_adv + loss_g_l1
            loss_g.backward()
            optimizer_g.step()
            
            epoch_d_loss += loss_d.item()
            epoch_g_loss += loss_g.item()
            num_batches += 1
        
        avg_d = epoch_d_loss / num_batches
        avg_g = epoch_g_loss / num_batches
        print(f"Epoch {epoch+1}/{epochs} - D Loss: {avg_d:.4f} - G Loss: {avg_g:.4f}")
        log_file.write(f"{epoch+1},{avg_d:.4f},{avg_g:.4f}\n")
    
    log_file.close()
    
    # Save models
    torch.save(G.state_dict(), "generator_speech_enhancement.pth")
    torch.save(D.state_dict(), "discriminator_speech_enhancement.pth")
    print("Models saved: generator_speech_enhancement.pth and discriminator_speech_enhancement.pth")
    print("Training log saved: training_log.txt")


# Run training and testing

train_gan(train_dir)


## Test

In [ ]:
# Test function with spectrogram
def test_gan(test_file, generator_path="generator_speech_enhancement.pth", segment_length=16384):
    G = Generator().to(device)
    G.load_state_dict(torch.load(generator_path, map_location=device))
    G.eval()
    
    fs, data = wavfile.read(test_file)
    noisy = data[:, 0].astype(np.float32) / 32768.0
    clean = data[:, 1].astype(np.float32) / 32768.0 if data.shape[1] > 1 else None
    
    # Force same length as training
    length = len(noisy)
    if length > segment_length:
        # For demo: take first segment (or random, or process in chunks)
        noisy = noisy[:segment_length]
        if clean is not None:
            clean = clean[:segment_length]
    else:
        pad = segment_length - length
        noisy = np.pad(noisy, (0, pad), mode='constant')
        if clean is not None:
            clean = np.pad(clean, (0, pad), mode='constant')
    
    with torch.no_grad():
        noisy_tensor = torch.tensor(noisy, dtype=torch.float32).unsqueeze(0).to(device)  # [1, 16384]
        z = torch.randn(1, 1024, 8, device=device)   # correct shape for 16384 / 2048 = 8
        enhanced = G(noisy_tensor, z).squeeze(0).cpu().numpy()   # [16384]
    
    # Save (trim padding if desired)
    enhanced_int16 = (enhanced * 32768.0).astype(np.int16)
    wavfile.write("enhanced_output.wav", fs, enhanced_int16)
    print("Enhanced audio saved as: enhanced_output.wav")
    
    # Show spectrograms (will use full 16384 samples)
    plot_spectrograms(noisy, enhanced, fs=fs, title="Noisy vs Enhanced Spectrogram")
    
    # Time-domain plot
    plt.figure(figsize=(12, 4))
    plt.plot(noisy, label='Noisy', alpha=0.7)
    plt.plot(enhanced, label='Enhanced', alpha=0.7)
    plt.legend()
    plt.title("Time-domain comparison")
    plt.show()
    
    # Metrics if clean is available
    if clean is not None:
        min_len = min(len(clean), len(enhanced))
        clean_trim = clean[:min_len]
        enhanced_trim = enhanced[:min_len]
        
        try:
            from pesq import pesq
            from pystoi import stoi
            pesq_score = pesq(fs, clean_trim, enhanced_trim, 'wb')
            stoi_score = stoi(clean_trim, enhanced_trim, fs, extended=False)
            print(f"PESQ: {pesq_score:.2f}")
            print(f"STOI: {stoi_score:.2f}")
        except ImportError:
            print("PESQ/STOI not available")
        
        sisdr_score = compute_sisdr(clean_trim, enhanced_trim)
        print(f"SISDR: {sisdr_score:.2f} dB")


# SISDR implementation (pure Python/Numpy)
def compute_sisdr(reference, estimation):
    reference = reference - np.mean(reference)
    estimation = estimation - np.mean(estimation)
    alpha = np.dot(estimation, reference) / (np.dot(reference, reference) + 1e-8)
    target = alpha * reference
    noise = estimation - target
    return 10 * np.log10((np.dot(target, target) + 1e-8) / (np.dot(noise, noise) + 1e-8))


def plot_spectrograms(noisy, enhanced, fs=16000, title="Spectrograms"):
    """
    Plot magnitude spectrograms of noisy and enhanced signals side by side
    """
    # Compute STFT
    _, _, S_noisy = stft(noisy, fs=fs, nperseg=512, noverlap=256)
    _, _, S_enhanced = stft(enhanced, fs=fs, nperseg=512, noverlap=256)
    
    # Magnitude in dB
    S_noisy_db = 20 * np.log10(np.abs(S_noisy) + 1e-8)
    S_enhanced_db = 20 * np.log10(np.abs(S_enhanced) + 1e-8)
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 5))
    
    im1 = axs[0].pcolormesh(S_noisy_db, shading='gouraud', cmap='viridis')
    axs[0].set_title('Noisy Spectrogram')
    axs[0].set_ylabel('Frequency [Hz]')
    axs[0].set_xlabel('Time [s]')
    fig.colorbar(im1, ax=axs[0], label='Magnitude (dB)')
    
    im2 = axs[1].pcolormesh(S_enhanced_db, shading='gouraud', cmap='viridis')
    axs[1].set_title('Enhanced Spectrogram')
    axs[1].set_ylabel('Frequency [Hz]')
    axs[1].set_xlabel('Time [s]')
    fig.colorbar(im2, ax=axs[1], label='Magnitude (dB)')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()



test_gan(test_file)